In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------- CONFIG ----------------
FILES = {
    "baseline_untargeted": "results_eval.txt",
    "en_untargeted": "results_eval_en.txt",
    "nn_untargeted": "results_eval_nn.txt",
    "nn_en_untargeted": "results_eval_nn_en.txt",  # NEW

    "baseline_targeted": "results_eval_tar.txt",
    "en_targeted": "results_eval_en_tar.txt",
    "nn_targeted": "results_eval_nn_tar.txt",
}

OUTPUT_DIR = "plots"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# ---------------------------------------


def safe_split(line):
    parts = []
    current = ""
    bracket = 0

    for c in line:
        if c == "[":
            bracket += 1
        elif c == "]":
            bracket -= 1

        if c == "," and bracket == 0:
            parts.append(current.strip())
            current = ""
        else:
            current += c

    parts.append(current.strip())
    return parts


def parse_file(path):
    rows = []
    with open(path, "r") as f:
        next(f)
        for line in f:
            parts = safe_split(line.strip())
            if len(parts) < 7:
                continue

            rows.append({
                "test": parts[0],
                "param": parts[1],
                "resnet34": float(parts[3]) * 100
            })

    return pd.DataFrame(rows)


def simplify_name(test, param):
    if "brightness" in test:
        return f"Brightness ({param})"
    elif "contrast" in test:
        return f"Contrast ({param})"
    elif "JPEG" in test:
        return f"JPEG ({param})"
    elif "Gaussian_blur" in test:
        return f"Blur {param}"
    elif "resize" in test:
        return f"Resize ({param})"
    elif "rotate" in test:
        return f"Rotate ({param})"
    elif "perspective" in test:
        return f"Perspective {param}"
    else:
        return test


def prepare(df):
    df = df.copy()
    df["label"] = df.apply(lambda x: simplify_name(x["test"], x["param"]), axis=1)
    return df


def plot_group_untargeted(base, en, nn, nn_en):
    labels = base["label"]
    x = range(len(labels))
    width = 0.2

    plt.figure(figsize=(18, 7))

    plt.bar([i - 1.5*width for i in x], base["resnet34"], width, label="Baseline")
    plt.bar([i - 0.5*width for i in x], en["resnet34"], width, label="EN")
    plt.bar([i + 0.5*width for i in x], nn["resnet34"], width, label="NN")
    plt.bar([i + 1.5*width for i in x], nn_en["resnet34"], width, label="EN+NN")

    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("ASR (%)")
    plt.title("Untargeted ASR across Transformations (ResNet34)")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/untargeted_resnet34.png", dpi=400)
    plt.close()


def plot_group_targeted(base, en, nn):
    labels = base["label"]
    x = range(len(labels))
    width = 0.25

    plt.figure(figsize=(18, 7))

    plt.bar([i - width for i in x], base["resnet34"], width, label="Baseline")
    plt.bar(x, en["resnet34"], width, label="EN")
    plt.bar([i + width for i in x], nn["resnet34"], width, label="NN")

    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("ASR (%)")
    plt.title("Targeted ASR across Transformations (ResNet34)")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/targeted_resnet34.png", dpi=400)
    plt.close()


# ---------------- MAIN ----------------

data = {k: prepare(parse_file(v)) for k, v in FILES.items()}

# Untargeted (4 bars)
plot_group_untargeted(
    data["baseline_untargeted"],
    data["en_untargeted"],
    data["nn_untargeted"],
    data["nn_en_untargeted"]
)

# Targeted (3 bars)
plot_group_targeted(
    data["baseline_targeted"],
    data["en_targeted"],
    data["nn_targeted"]
)

print("Plots generated successfully.")

Plots generated successfully.
